In [33]:
import polars as pl

# define the schema of the dataframe
event_schema = pl.Struct({"aid": pl.Int32, "ts": pl.Int64, "type": str})
df_schema = {"session": pl.Int32, "events": pl.List(event_schema)}

df = pl.read_ndjson('../data/train.jsonl', schema=df_schema, low_memory=True)

print(df.head())

shape: (5, 2)
┌─────────┬─────────────────────────────────┐
│ session ┆ events                          │
│ ---     ┆ ---                             │
│ i32     ┆ list[struct[3]]                 │
╞═════════╪═════════════════════════════════╡
│ 0       ┆ [{1517085,1659304800025,"click… │
│ 1       ┆ [{424964,1659304800025,"carts"… │
│ 2       ┆ [{763743,1659304800038,"clicks… │
│ 3       ┆ [{1425967,1659304800095,"carts… │
│ 4       ┆ [{613619,1659304800119,"clicks… │
└─────────┴─────────────────────────────────┘


In [115]:
# amount of events per session
session_events_count = (
    df
    .select(pl.col("events"))
    .with_columns(session_events_count = pl.col("events").list.len())
    .sort("session_events_count")
    .select("session_events_count")
)

print(session_events_count)
print(session_events_count.describe())

shape: (12_899_779, 1)
┌──────────────────────┐
│ session_events_count │
│ ---                  │
│ u32                  │
╞══════════════════════╡
│ 2                    │
│ 2                    │
│ 2                    │
│ 2                    │
│ 2                    │
│ …                    │
│ 498                  │
│ 499                  │
│ 499                  │
│ 500                  │
│ 500                  │
└──────────────────────┘
shape: (9, 2)
┌────────────┬──────────────────────┐
│ statistic  ┆ session_events_count │
│ ---        ┆ ---                  │
│ str        ┆ f64                  │
╞════════════╪══════════════════════╡
│ count      ┆ 1.2899779e7          │
│ null_count ┆ 0.0                  │
│ mean       ┆ 16.799985            │
│ std        ┆ 33.57738             │
│ min        ┆ 2.0                  │
│ 25%        ┆ 3.0                  │
│ 50%        ┆ 6.0                  │
│ 75%        ┆ 15.0                 │
│ max        ┆ 500.0                │
└─────

In [93]:
# count amount of each event type
events_count = (
    df
    .select(pl.col("events"))
    .explode("events")
    .unnest("events")
    .select(["type"])
    .rename({"type": "event_type"})
    .group_by("event_type")
    .agg(
        pl.count("event_type")
        .alias("count")
    )
)

print(events_count)

shape: (3, 2)
┌────────────┬───────────┐
│ event_type ┆ count     │
│ ---        ┆ ---       │
│ str        ┆ u32       │
╞════════════╪═══════════╡
│ orders     ┆ 5098951   │
│ carts      ┆ 16896191  │
│ clicks     ┆ 194720954 │
└────────────┴───────────┘


In [117]:
# count amount of events per aid
events_per_aid = (
    df
    .select(pl.col("events"))
    .explode("events")
    .unnest("events")
    .select(["aid"])
    .group_by("aid")
    .agg(
        pl.count("aid")
        .alias("events_count")
    )
    .sort("events_count")
)

print(events_per_aid)
print(events_per_aid.select("events_count").describe())

shape: (1_855_603, 2)
┌─────────┬──────────────┐
│ aid     ┆ events_count │
│ ---     ┆ ---          │
│ i32     ┆ u32          │
╞═════════╪══════════════╡
│ 1726329 ┆ 3            │
│ 1166769 ┆ 3            │
│ 1164783 ┆ 3            │
│ 270567  ┆ 3            │
│ 679262  ┆ 3            │
│ …       ┆ …            │
│ 1733943 ┆ 105091       │
│ 29735   ┆ 113279       │
│ 108125  ┆ 118524       │
│ 485256  ┆ 126836       │
│ 1460571 ┆ 129004       │
└─────────┴──────────────┘
shape: (9, 2)
┌────────────┬──────────────┐
│ statistic  ┆ events_count │
│ ---        ┆ ---          │
│ str        ┆ f64          │
╞════════════╪══════════════╡
│ count      ┆ 1.855603e6   │
│ null_count ┆ 0.0          │
│ mean       ┆ 116.790119   │
│ std        ┆ 728.854992   │
│ min        ┆ 3.0          │
│ 25%        ┆ 9.0          │
│ 50%        ┆ 20.0         │
│ 75%        ┆ 56.0         │
│ max        ┆ 129004.0     │
└────────────┴──────────────┘


In [109]:
# Count the amount of sessions
sessions_count = df.height

print(sessions_count)

12899779


In [110]:
# Count unique items
items_count = (
    df
    .select(pl.col("events"))
    .explode("events")
    .unnest("events")
    .select("aid")
    .unique()
    .height
)

print(items_count)

1855603


In [118]:
# session length is counted as the last_ts - first_ts
session_length_hours = (
    df
    .with_row_index()
    .select(["index", "events"])
    .with_columns(first_event = pl.col("events").list[0])
    .unnest("first_event")
    .rename({"ts": "first_ts"})
    .drop(["aid", "type"])
    .with_columns(last_event = pl.col("events").list[-1])
    .unnest("last_event")
    .rename({"ts": "last_ts"})
    .drop(["aid", "type", "events"])
    .with_columns(session_length_hours = (pl.col("last_ts") - pl.col("first_ts")) / 3_600_000)
    .select(["index", "session_length_hours"])
    .sort("session_length_hours")
)

print(session_length_hours)
print(session_length_hours.select("session_length_hours").describe())

shape: (12_899_779, 2)
┌────────┬──────────────────────┐
│ index  ┆ session_length_hours │
│ ---    ┆ ---                  │
│ u32    ┆ f64                  │
╞════════╪══════════════════════╡
│ 34854  ┆ 0.0                  │
│ 34857  ┆ 0.0                  │
│ 50748  ┆ 0.0                  │
│ 98119  ┆ 0.0                  │
│ 113762 ┆ 0.0                  │
│ …      ┆ …                    │
│ 803    ┆ 671.98597            │
│ 914    ┆ 671.98716            │
│ 1527   ┆ 671.99134            │
│ 569    ┆ 671.993459           │
│ 37     ┆ 671.997421           │
└────────┴──────────────────────┘
shape: (9, 2)
┌────────────┬──────────────────────┐
│ statistic  ┆ session_length_hours │
│ ---        ┆ ---                  │
│ str        ┆ f64                  │
╞════════════╪══════════════════════╡
│ count      ┆ 1.2899779e7          │
│ null_count ┆ 0.0                  │
│ mean       ┆ 164.59407            │
│ std        ┆ 202.485156           │
│ min        ┆ 0.0                  │
│ 25%

In [80]:
# Example of long session with many events and multiple orders
events = df.row(37)[1]
for event in events:
    print(event)

{'aid': 1639625, 'ts': 1659304800543, 'type': 'clicks'}
{'aid': 1639625, 'ts': 1659304809946, 'type': 'clicks'}
{'aid': 1001553, 'ts': 1659304813870, 'type': 'clicks'}
{'aid': 509070, 'ts': 1659304818202, 'type': 'clicks'}
{'aid': 664490, 'ts': 1659304875246, 'type': 'clicks'}
{'aid': 959208, 'ts': 1659304889514, 'type': 'clicks'}
{'aid': 1470083, 'ts': 1659304899657, 'type': 'carts'}
{'aid': 311240, 'ts': 1659304911205, 'type': 'clicks'}
{'aid': 133198, 'ts': 1659304919343, 'type': 'clicks'}
{'aid': 350915, 'ts': 1659304929661, 'type': 'clicks'}
{'aid': 1660020, 'ts': 1659305025161, 'type': 'orders'}
{'aid': 934266, 'ts': 1659305025161, 'type': 'orders'}
{'aid': 1839606, 'ts': 1659305025161, 'type': 'orders'}
{'aid': 1470083, 'ts': 1659305025161, 'type': 'orders'}
{'aid': 1639625, 'ts': 1659305025161, 'type': 'orders'}
{'aid': 1097946, 'ts': 1659305025161, 'type': 'orders'}
{'aid': 565802, 'ts': 1659305025161, 'type': 'orders'}
{'aid': 576646, 'ts': 1659305025161, 'type': 'orders'}
{'